# Organ Classification in Abdominal CT Scans using Transfer Learning

**Course:** Deep Learning for Computer Vision (Project)
**Name:** ____________________ **Roll No.:** ____________________

### Aim
Train a convolutional neural network to identify which organ is shown in an abdominal CT slice. The model is
then used in a Streamlit application ("Doctor vs AI"), where a user competes with the model at the same task.

### Approach
- Dataset: **OrganAMNIST** from the MedMNIST v2 benchmark (real CT scans, 11 organ classes)
- Model: **ResNet18** pretrained on ImageNet, fine-tuned on the CT images (transfer learning)
- Training with **mixed precision (FP16)** on a GPU, data augmentation and regularisation
- Evaluation with accuracy, precision, recall, F1 score and a confusion matrix
- Model interpretation with **Grad-CAM**
- Experiments comparing mixed precision vs FP32, pretrained vs from scratch, and with vs without augmentation

### Syllabus topics covered
| Topic | Section |
|---|---|
| Image classification with CNN architectures (ResNet) | 5 |
| Transfer learning | 5, 10 |
| Mixed precision training | 6, 10 |
| Data pre-processing and augmentation | 4 |
| Loss functions, optimisation, regularisation, hyperparameters | 6 |
| Batch normalisation, activation functions | 5 |
| Metrics | 8 |
| Medical imaging | whole notebook |

**Runtime:** Google Colab, GPU runtime (Runtime → Change runtime type → T4 GPU).

## 1. Environment setup

In [ ]:
import os
import json
import math
import time
import copy
import random
import shutil
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset

**OrganAMNIST** is part of MedMNIST v2 (Yang et al., *Scientific Data*, 2023). The images are axial slices from 3D
abdominal CT scans of the LiTS (Liver Tumor Segmentation) benchmark, cropped around the bounding box of one of
11 organs. We use the 64×64 version from MedMNIST+ and the official train / validation / test split.

- License: CC BY 4.0
- Source: https://medmnist.com/ (files hosted on Zenodo)

In [ ]:
DATA_URL = "https://zenodo.org/api/records/10519652/files/organamnist_64.npz/content"
DATA_PATH = "data/organamnist_64.npz"

os.makedirs("data", exist_ok=True)
if not os.path.exists(DATA_PATH):
    print("Downloading OrganAMNIST (about 200 MB)...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

data = np.load(DATA_PATH)
x_train, y_train = data["train_images"], data["train_labels"].ravel().astype(np.int64)
x_val, y_val = data["val_images"], data["val_labels"].ravel().astype(np.int64)
x_test, y_test = data["test_images"], data["test_labels"].ravel().astype(np.int64)

CLASSES = ["bladder", "femur-left", "femur-right", "heart", "kidney-left", "kidney-right",
           "liver", "lung-left", "lung-right", "pancreas", "spleen"]
NUM_CLASSES = len(CLASSES)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "images": [len(x_train), len(x_val), len(x_test)],
    "image shape": [x_train.shape[1:], x_val.shape[1:], x_test.shape[1:]],
    "dtype": [x_train.dtype, x_val.dtype, x_test.dtype],
})

## 3. Exploratory data analysis

### 3.1 Class distribution

In [ ]:
counts = pd.DataFrame({
    "train": np.bincount(y_train, minlength=NUM_CLASSES),
    "validation": np.bincount(y_val, minlength=NUM_CLASSES),
    "test": np.bincount(y_test, minlength=NUM_CLASSES),
}, index=CLASSES)
counts["train %"] = (counts["train"] / counts["train"].sum() * 100).round(1)
display(counts)

counts["train"].plot.bar(figsize=(10, 4), color="steelblue", edgecolor="black")
plt.title("Number of training images per organ")
plt.ylabel("images")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

The classes are **imbalanced**. Liver has about 6,000 training images, while each femur class has fewer than 1,400.
Accuracy alone can hide poor performance on the small classes, so we also report the **macro-averaged F1 score**,
which weights every class equally.

### 3.2 Sample images from each class

In [ ]:
fig, axes = plt.subplots(4, NUM_CLASSES, figsize=(18, 7))
for c in range(NUM_CLASSES):
    idx = np.random.choice(np.where(y_train == c)[0], 4, replace=False)
    for r in range(4):
        ax = axes[r, c]
        ax.imshow(x_train[idx[r]], cmap="gray")
        ax.axis("off")
        if r == 0:
            ax.set_title(CLASSES[c], fontsize=10)
plt.suptitle("Random training samples (4 per class)", fontsize=14)
plt.tight_layout()
plt.show()

CT images follow the **radiological convention**: the patient's left side is shown on the right side of the image.
This matters for the left/right classes (femur, kidney, lung). The model has to use the position of the organ in the
slice, not just its shape. Because of this, **horizontal flipping cannot be used as an augmentation**: it would turn
a left kidney into a right kidney without changing the label.

### 3.3 Pixel intensity statistics

In [ ]:
print(f"Pixel range: {x_train.min()} to {x_train.max()}")
print(f"Mean: {x_train.mean():.2f}   Std: {x_train.std():.2f}")

plt.figure(figsize=(8, 3.5))
plt.hist(x_train[:5000].ravel(), bins=64, color="gray")
plt.title("Pixel intensity histogram (5,000 training images)")
plt.xlabel("intensity")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## 4. Pre-processing and data augmentation

**Pre-processing**
1. Scale pixel values from [0, 255] to [0, 1].
2. Repeat the single grayscale channel 3 times, because the ImageNet-pretrained network expects RGB input.
3. Normalise with the ImageNet mean and standard deviation, so the inputs have the same statistics the pretrained weights were trained on.

**Augmentation** (training only): random rotation of ±10°, scaling of 0.9–1.1× and translation of ±8%.
These simulate small differences in patient position and scanner field of view. No flips are used (see 3.2).

The whole dataset is small (about 140 MB as uint8), so it is moved to the GPU once and augmentation is done on the GPU
with `affine_grid` / `grid_sample`. This is much faster than a CPU DataLoader for images of this size.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def preprocess(x_uint8):
    # (N, 64, 64) uint8 -> (N, 3, 64, 64) normalised float
    x = x_uint8.float().div(255).unsqueeze(1).repeat(1, 3, 1, 1)
    return (x - IMAGENET_MEAN) / IMAGENET_STD


def augment(x, max_rot_deg=10, scale_range=0.1, max_shift=0.08):
    n = x.size(0)
    angle = (torch.rand(n, device=x.device) * 2 - 1) * math.radians(max_rot_deg)
    scale = 1 + (torch.rand(n, device=x.device) * 2 - 1) * scale_range
    tx = (torch.rand(n, device=x.device) * 2 - 1) * max_shift
    ty = (torch.rand(n, device=x.device) * 2 - 1) * max_shift
    cos, sin = torch.cos(angle) / scale, torch.sin(angle) / scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1),
                         torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="border", align_corners=False)


# keep the data on the GPU as uint8 and convert batch by batch
Xtr, Ytr = torch.from_numpy(x_train).to(device), torch.from_numpy(y_train).to(device)
Xva, Yva = torch.from_numpy(x_val).to(device), torch.from_numpy(y_val).to(device)
Xte, Yte = torch.from_numpy(x_test).to(device), torch.from_numpy(y_test).to(device)
print("Train tensor:", Xtr.shape, Xtr.dtype, Xtr.device)

In [ ]:
# visualise a few augmented versions of the same image
sample = Xtr[:1].repeat(8, 1, 1)
aug = augment(preprocess(sample))
aug_img = (aug * IMAGENET_STD + IMAGENET_MEAN)[:, 0].clamp(0, 1).cpu().numpy()

fig, axes = plt.subplots(1, 9, figsize=(16, 2.4))
axes[0].imshow(x_train[0], cmap="gray")
axes[0].set_title("original")
for i in range(8):
    axes[i + 1].imshow(aug_img[i], cmap="gray")
    axes[i + 1].set_title(f"augmented {i + 1}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Model: ResNet18 with transfer learning

**ResNet18** (He et al., 2016) is made of residual blocks. Each block learns a residual `F(x)` and adds it to its input
(`y = F(x) + x`). These skip connections let gradients flow through deep networks and avoid the degradation problem
seen in plain deep CNNs such as VGG. Every convolution is followed by **batch normalisation** and a **ReLU** activation.

**Transfer learning:** we start from weights pretrained on ImageNet (1.2M natural images). The early layers already
detect edges, textures and shapes, which are also useful for CT images. All layers are fine-tuned, and only the final
fully-connected layer is replaced with a new layer that has 11 outputs.

**Change for small images:** the standard ResNet reduces a 224×224 image by 32× (stride-2 first convolution, then max-pooling).
A 64×64 input would end up as a 2×2 feature map. We set the first convolution's stride to 1 and remove the max-pool layer.
This gives an 8×8 feature map at the last block, which keeps more spatial detail for classification and for Grad-CAM.
The pretrained weights are unchanged by this, because stride and pooling have no parameters.

In [ ]:
def build_model(pretrained=True):
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.conv1.stride = (1, 1)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model


model = build_model(pretrained=True).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_params / 1e6:.2f} M")

# check the size of the feature map produced by each stage
x = preprocess(Xtr[:2])
with torch.no_grad():
    x = model.relu(model.bn1(model.conv1(x)))
    print(f"{'stem':8s}", tuple(x.shape))
    for name in ["layer1", "layer2", "layer3", "layer4"]:
        x = getattr(model, name)(x)
        print(f"{name:8s}", tuple(x.shape))

## 6. Training

| Component | Choice | Reason |
|---|---|---|
| Loss | Cross-entropy with label smoothing (0.1) | Standard multi-class loss. Smoothing keeps the model from becoming over-confident |
| Optimiser | AdamW, lr = 1e-3, weight decay = 1e-4 | Adam with decoupled weight decay (L2-style regularisation) |
| LR schedule | One-Cycle | Warm-up followed by cosine decay, which converges well in few epochs |
| Batch size | 128 | |
| Epochs | 8 | Validation accuracy levels off after this |
| Precision | Mixed (FP16 autocast + GradScaler) | Faster training and lower memory on Tensor-Core GPUs |

**Mixed precision:** inside `torch.autocast`, matrix multiplications and convolutions run in FP16, while operations that are
sensitive to precision (softmax, loss, reductions) stay in FP32. A master copy of the weights is kept in FP32.
`GradScaler` multiplies the loss by a large factor before backpropagation so that small gradients don't underflow to zero
in FP16, then un-scales them before the optimiser step.

The model with the best validation accuracy is kept, which works as a simple form of early stopping.

In [ ]:
EPOCHS = 8
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1


def iterate_batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def evaluate(model, x, y, use_amp=True):
    model.eval()
    total_loss, all_probs = 0.0, []
    for xb, yb in iterate_batches(x, y, 512, shuffle=False):
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp and device.type == "cuda"):
            logits = model(preprocess(xb))
        total_loss += F.cross_entropy(logits.float(), yb, reduction="sum").item()
        all_probs.append(logits.float().softmax(1))
    probs = torch.cat(all_probs)
    acc = (probs.argmax(1) == y).float().mean().item()
    return total_loss / len(x), acc, probs.cpu().numpy()


def train_model(pretrained=True, use_amp=True, use_aug=True, epochs=EPOCHS, verbose=True):
    torch.manual_seed(SEED)
    model = build_model(pretrained).to(device)
    amp = use_amp and device.type == "cuda"

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(Xtr) / BATCH_SIZE)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=epochs * steps_per_epoch)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)

    history, best_acc, best_state = [], 0.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        start = time.perf_counter()
        seen, correct, loss_sum = 0, 0, 0.0

        for xb, yb in iterate_batches(Xtr, Ytr, BATCH_SIZE, shuffle=True):
            xb = preprocess(xb)
            if use_aug:
                xb = augment(xb)
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                logits = model(xb)
                loss = criterion(logits, yb)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            seen += len(yb)
            loss_sum += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()

        sync()
        epoch_time = time.perf_counter() - start
        val_loss, val_acc, _ = evaluate(model, Xva, Yva, use_amp)
        history.append(dict(epoch=epoch, train_loss=loss_sum / seen, train_acc=correct / seen,
                            val_loss=val_loss, val_acc=val_acc, secs=epoch_time))
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch:2d}/{epochs} | train loss {loss_sum / seen:.4f} acc {correct / seen:.4f} | "
                  f"val loss {val_loss:.4f} acc {val_acc:.4f} | {epoch_time:.1f}s")

    model.load_state_dict(best_state)
    return model, history

In [ ]:
model, history = train_model(pretrained=True, use_amp=True, use_aug=True, epochs=EPOCHS)

os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/model.pt")
print("Best validation accuracy:", max(h["val_acc"] for h in history))

## 7. Training curves

In [ ]:
hist = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(hist.epoch, hist.train_loss, "o-", label="train")
ax1.plot(hist.epoch, hist.val_loss, "o-", label="validation")
ax1.set_title("Loss")
ax1.set_xlabel("epoch")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(hist.epoch, hist.train_acc, "o-", label="train")
ax2.plot(hist.epoch, hist.val_acc, "o-", label="validation")
ax2.set_title("Accuracy")
ax2.set_xlabel("epoch")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

hist.round(4)

Note: training accuracy is computed on **augmented** images and the training loss includes **label smoothing**, so
training accuracy can be slightly lower than validation accuracy. That doesn't mean the model is under-fitting. The
signal to watch is the validation loss: if it starts rising while the training loss keeps falling, the model is over-fitting.

## 8. Evaluation on the test set

### 8.1 Overall metrics

In [ ]:
test_loss, test_acc, test_probs = evaluate(model, Xte, Yte)
test_pred = test_probs.argmax(1)
macro_f1 = f1_score(y_test, test_pred, average="macro")

print(f"Test accuracy : {test_acc:.4f}")
print(f"Test loss     : {test_loss:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print()
print(classification_report(y_test, test_pred, target_names=CLASSES, digits=4))

For reference, the MedMNIST v2 paper reports **93.5%** test accuracy for ResNet-18 at 28×28 resolution and **95.1%**
at 224×224 on OrganAMNIST.

### 8.2 Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cm_norm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] > 0:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm_norm[i, j] > 0.5 else "black")
plt.colorbar(im, fraction=0.046)
plt.title("Confusion matrix (test set, colour = row-normalised)")
plt.tight_layout()
plt.show()

# most frequent mistakes
mistakes = [(CLASSES[i], CLASSES[j], cm[i, j]) for i in range(NUM_CLASSES)
            for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
pd.DataFrame(sorted(mistakes, key=lambda t: -t[2])[:8], columns=["true", "predicted", "count"])

### 8.3 Misclassified examples

In [ ]:
wrong = np.where(test_pred != y_test)[0]
print(f"{len(wrong)} of {len(y_test)} test images misclassified")

pick = np.random.choice(wrong, 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(x_test[i], cmap="gray")
    ax.set_title(f"true: {CLASSES[y_test[i]]}\npred: {CLASSES[test_pred[i]]} ({test_probs[i].max():.2f})",
                 fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9. Model interpretation with Grad-CAM

Grad-CAM (Selvaraju et al., 2017) shows which parts of the image the network used for its prediction:
1. Take the activations `A` of the last convolutional block (`layer4`, 512 channels × 8×8).
2. Compute the gradient of the predicted class score with respect to `A`.
3. Average the gradients over the spatial dimensions to get one importance weight per channel.
4. Take the weighted sum of the activation maps, apply ReLU, and upsample to the image size.

In [ ]:
def grad_cam(model, img_uint8):
    model.eval()
    store = {}
    h1 = model.layer4.register_forward_hook(lambda m, i, o: store.update(act=o))
    h2 = model.layer4.register_full_backward_hook(lambda m, gi, go: store.update(grad=go[0]))

    x = preprocess(torch.from_numpy(img_uint8[None]).to(device))
    logits = model(x)
    cls = logits.argmax(1).item()
    model.zero_grad()
    logits[0, cls].backward()
    h1.remove()
    h2.remove()

    weights = store["grad"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((weights * store["act"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=img_uint8.shape, mode="bilinear", align_corners=False)[0, 0]
    cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy(), cls


fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for k, c in enumerate([0, 3, 4, 5, 6, 9]):
    i = np.random.choice(np.where(y_test == c)[0])
    cam, pred = grad_cam(model, x_test[i])
    axes[0, k].imshow(x_test[i], cmap="gray")
    axes[0, k].set_title(f"true: {CLASSES[y_test[i]]}", fontsize=10)
    axes[1, k].imshow(x_test[i], cmap="gray")
    axes[1, k].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, k].set_title(f"pred: {CLASSES[pred]}", fontsize=10)
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Grad-CAM: regions that contributed most to the prediction", fontsize=13)
plt.tight_layout()
plt.show()

## 10. Experiments

### 10.1 Mixed precision vs FP32: training speed and memory
The same training step (forward, backward and optimiser update on one batch of 128) is timed in both modes.
The first 5 steps are a warm-up and are not timed.

In [ ]:
def benchmark(use_amp, steps=30, batch_size=128):
    torch.manual_seed(SEED)
    bench_model = build_model().to(device)
    opt = torch.optim.AdamW(bench_model.parameters(), lr=LR)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    xb, yb = preprocess(Xtr[:batch_size]), Ytr[:batch_size]
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for step in range(steps + 5):
        if step == 5:
            sync()
            start = time.perf_counter()
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(bench_model(xb), yb)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    sync()
    ms = (time.perf_counter() - start) / steps * 1000
    mem = torch.cuda.max_memory_allocated() / 1024**2 if device.type == "cuda" else float("nan")
    del bench_model, opt
    return ms, mem


fp32_ms, fp32_mem = benchmark(use_amp=False)
amp_ms, amp_mem = benchmark(use_amp=device.type == "cuda")

bench = pd.DataFrame({"ms per step": [fp32_ms, amp_ms], "peak GPU memory (MB)": [fp32_mem, amp_mem]},
                     index=["FP32", "Mixed precision (FP16)"]).round(1)
display(bench)
print(f"Speed-up with mixed precision: {fp32_ms / amp_ms:.2f}x")

bench["ms per step"].plot.barh(figsize=(7, 2.5), color=["gray", "seagreen"])
plt.xlabel("milliseconds per training step (lower is better)")
plt.tight_layout()
plt.show()

### 10.2 Ablation study
To see what each technique contributes, four short runs (3 epochs each, same seed) are compared:

| Run | Pretrained | Mixed precision | Augmentation |
|---|---|---|---|
| baseline | yes | yes | yes |
| no_amp | yes | **no** | yes |
| from_scratch | **no** | yes | yes |
| no_augmentation | yes | yes | **no** |

In [ ]:
EXP_EPOCHS = 3
runs = {
    "baseline": dict(pretrained=True, use_amp=True, use_aug=True),
    "no_amp": dict(pretrained=True, use_amp=False, use_aug=True),
    "from_scratch": dict(pretrained=False, use_amp=True, use_aug=True),
    "no_augmentation": dict(pretrained=True, use_amp=True, use_aug=False),
}

results = []
for name, cfg in runs.items():
    print(f"--- {name} ---")
    m, h = train_model(epochs=EXP_EPOCHS, verbose=False, **cfg)
    _, acc, probs = evaluate(m, Xte, Yte)
    f1 = f1_score(y_test, probs.argmax(1), average="macro")
    sec = float(np.mean([r["secs"] for r in h]))
    print(f"test acc {acc:.4f} | macro F1 {f1:.4f} | {sec:.1f} s/epoch")
    results.append(dict(run=name, test_acc=acc, macro_f1=f1, sec_per_epoch=sec))

    out = f"artifacts/experiments/{name}"
    os.makedirs(out, exist_ok=True)
    json.dump(h, open(f"{out}/history.json", "w"), indent=2)
    json.dump(dict(test_acc=acc, macro_f1=f1, config=cfg), open(f"{out}/metrics.json", "w"), indent=2)
    del m

exp = pd.DataFrame(results).set_index("run")
exp.round(4)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5))
exp["test_acc"].plot.barh(ax=ax1, color="steelblue")
ax1.set_xlim(exp["test_acc"].min() - 0.05, 1.0)
ax1.set_title("Test accuracy after 3 epochs")
exp["sec_per_epoch"].plot.barh(ax=ax2, color="darkorange")
ax2.set_title("Seconds per epoch")
plt.tight_layout()
plt.show()

**What to look for:**
- *no_amp* vs *baseline*: similar accuracy, but each epoch is slower. Mixed precision speeds up training without hurting accuracy.
- *from_scratch* vs *baseline*: starting from random weights gives lower accuracy in the same number of epochs, which shows the value of transfer learning.
- *no_augmentation* vs *baseline*: augmentation does not necessarily raise accuracy on this test set. The test slices are
  centred crops with the same framing as the training slices, and with only 3 epochs the extra variation mainly slows
  convergence. The benefit of augmentation is robustness to shifted, rotated or noisy inputs, which is tested separately
  in the Streamlit app.

## 11. Save results for the Streamlit application
The app (`app.py`) loads the trained weights and these metric files to build its report page.

In [ ]:
per_class = {}
report = classification_report(y_test, test_pred, target_names=CLASSES, output_dict=True)
for c in CLASSES:
    per_class[c] = dict(precision=report[c]["precision"], recall=report[c]["recall"],
                        f1=report[c]["f1-score"], support=int(report[c]["support"]))

metrics = dict(
    test_acc=test_acc,
    test_loss=test_loss,
    macro_f1=float(macro_f1),
    per_class=per_class,
    confusion_matrix=cm.tolist(),
    amp_benchmark_ms_per_step=dict(fp32=round(fp32_ms, 1), amp=round(amp_ms, 1)),
    config=dict(epochs=EPOCHS, bs=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
                label_smoothing=LABEL_SMOOTHING, device=str(device),
                model="ResNet18 (ImageNet pretrained)"),
)
json.dump(metrics, open("artifacts/metrics.json", "w"), indent=2)
json.dump(history, open("artifacts/history.json", "w"), indent=2)

shutil.make_archive("artifacts", "zip", ".", "artifacts")
print(sorted(os.listdir("artifacts")))
print(f"artifacts.zip: {os.path.getsize('artifacts.zip') / 1e6:.1f} MB")

In [ ]:
# download the results (Colab only). Unzip into the project folder next to app.py
try:
    from google.colab import files
    files.download("artifacts.zip")
except ImportError:
    print("Not running on Colab: artifacts.zip saved in", os.getcwd())

## 12. Conclusion

- An ImageNet-pretrained ResNet18 was fine-tuned on 34,561 real abdominal CT slices to classify 11 organs. Its test
  accuracy and macro F1 are reported in Section 8, alongside the published MedMNIST v2 benchmark.
- Most errors come from anatomically similar or neighbouring structures, especially the left/right pairs. This is
  expected, because the only cue there is the organ's position in the slice.
- Mixed precision reduced the time per training step and the GPU memory use without lowering accuracy (Section 10.1).
- Transfer learning improved test performance over training from scratch (Section 10.2). Augmentation did not improve
  accuracy on the clean test set in a 3-epoch run. Its role here is robustness to variation in the input rather than
  in-distribution accuracy.
- Grad-CAM shows the network mostly focuses on the organ region rather than the background, which makes its
  decisions easier to trust.

The trained model is used in the Streamlit application, which has these features:
- a game where the user and the model classify the same scans
- a robustness test with noise, blur and rotation
- Grad-CAM visualisation
- a report page with these metrics

### References
1. J. Yang et al., "MedMNIST v2: A large-scale lightweight benchmark for 2D and 3D biomedical image classification," *Scientific Data*, 10, 41, 2023.
2. K. He, X. Zhang, S. Ren, J. Sun, "Deep Residual Learning for Image Recognition," CVPR 2016.
3. R. R. Selvaraju et al., "Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization," ICCV 2017.
4. P. Micikevicius et al., "Mixed Precision Training," ICLR 2018.
5. P. Bilic et al., "The Liver Tumor Segmentation Benchmark (LiTS)," *Medical Image Analysis*, 84, 2023.